In [1]:
import re
import pandas as pd 
from collections import defaultdict
from typing import List, Dict, Tuple

In [2]:
train_text = """ I love Deep Learning. 
And I love NLP, with or without deep learning. 
In general I love AI, but I don't like A.B.C."""


In [3]:
def tokenize(text: str, pattern: str = None) -> List[str]:
    pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
    tokens = re.findall(pattern, text)
    return tokens


In [4]:
train_tokens = tokenize(train_text)
# train_tokens

In [5]:

def ngram_counts(tokens: List[str], n: int) -> Dict[Tuple[str, ...], Dict[str, int]]:
    """ 
    Compute the ngram counts of given tokens with specifed context size.
    
    Args:
        tokens: list of tokenized strings.
        n size of context, n=1 >> bigram, n=2 >> trigram etc. 

    Returns:
        Nested Dict of (context, next_word) counts.
    """
    counts = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens) - n):
        context = tuple(tokens[i: i + n])
        next_ = tokens[i+n]
        counts[context][next_] += 1
    return counts
    
    

In [6]:
unicnt = ngram_counts(train_tokens, n=0)
bicnt = ngram_counts(train_tokens, n=1)
tricnt = ngram_counts(train_tokens, n=2)


In [7]:
tricnt_df = pd.DataFrame.from_dict(tricnt, orient="index").fillna(0)
tricnt_df.T

,I,love,Deep,deep,Learning,.,In,AI,And,general,...,NLP,with,or,without,learning,.,love,but,I,don't
,love,Deep,Learning,learning,.,And,general,but,I,I,...,with,or,without,deep,.,In,AI,I,don't,like
Deep,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NLP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AI,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Learning,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
.,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
And,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
I,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
love,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
with,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
def coocurr(tokens: List[str], window_size: int) -> Dict[str, Dict[str, int]]:
    """ 
    Compute the co-occurrence counts of given tokens with specifed context (windon) size.
    
    Args:
        tokens: list of tokenized strings.
        window_size: size of context (window)

    Returns:
        Nested Dict of (context, next_word) counts.
    """
    counts = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens)):
        start = max(0, i-window_size)
        end = min(len(tokens), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                counts[tokens[i]][tokens[j]] += 1
    return counts
        

In [9]:
cooc = coocurr(train_tokens, 4)
cooc_df = pd.DataFrame.from_dict(cooc, orient="index").fillna(0)
cooc_df

    


,love,Deep,Learning,.,And,NLP,with,or,learning,In,general,AI,but,I,don't,like,A.B.C.,without,deep
I,4.0,2.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,0.0,0.0
Deep,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Learning,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
.,3.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,1.0
And,2.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
NLP,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
with,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
or,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
without,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
In,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0


In [10]:
def coocurr_efficient(tokens: List[str], window_size: int) -> Dict[str, Dict[str, int]]:
    """
    Compute the co-occurrence counts of given tokens with specified context (window) size.

    Args:
        tokens: list of tokenized strings.
        window_size: size of context (window)

    Returns:
        Nested Dict of (word, context_word) counts.

    Note: probably using gensim dictionary and lil_matrix from scipy is more efficient. 
    """
    counts = defaultdict(lambda: defaultdict(int))
    n = len(tokens)
    for i in range(n):
        w_i = tokens[i]
        inner = counts[w_i]  # caching to avoid repeating list indexing
        for j in range(i + 1, min(n, i + window_size + 1)):
            w_j = tokens[j]
            inner[w_j] += 1  # caching to avoid repeating list indexing
            counts[w_j][w_i] += 1
    return counts




In [11]:
cooc_efficient = coocurr_efficient(train_tokens, 4)
cooc_efficient_df = pd.DataFrame.from_dict(cooc_efficient, orient="index").fillna(0)
cooc_efficient_df

,love,Deep,Learning,.,And,NLP,with,or,learning,In,general,AI,but,I,don't,like,A.B.C.,without,deep
I,4.0,2.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,0.0,0.0
Deep,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Learning,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
.,3.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,1.0
And,2.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
NLP,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
with,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
or,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
without,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
In,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0


In [12]:
# Reindex both to the same sorted columns and index, fill missing with 0
cols = sorted(set(cooc_df.columns) | set(cooc_efficient_df.columns))
idx  = sorted(set(cooc_df.index)   | set(cooc_efficient_df.index))

a = cooc_df.reindex(index=idx, columns=cols).fillna(0)
b = cooc_efficient_df.reindex(index=idx, columns=cols).fillna(0)

# Check all values match
print(a.equals(b))          # True/False
print((a - b).abs().max().max())  # should be 0.0 if identical

True
0.0
